# 무역 트렌드와 산업생산의 대조 — 정본 계산기

`연구계획서.md`의 둘째 갈래. 성질별 수출 실적으로 직접 만든 품목군 월 계열에
국가데이터처 광공업생산지수(`fact_ip`)를 붙여 수출과 생산의 성장률·동조성·선후행·브리지 회귀를 잰다. 수출 척도는 명목과 코드 단위 Törnqvist 물량이고
(단가 = 명목 − 물량), 한국은행 수출물가지수(2026-09-03부터 `cache/fact_xmpi.parquet`, DB에서는 뺌)로 실질화한 값과 수출출하지수는 참고 열로만 둔다(논문에 없음).
대응표는 계획서 §VII.1 표 A(2026-09-02 확정, 논문 표 2)를 코드로 옮긴 것이다.

| 절 | 내용 |
|---|---|
| §0 | 설정. DB(읽기전용), 품목군 대응표 |
| §1 | 대응표(논문 표 2)와 생산지수 구축, 코드 단위 Törnqvist 물량. 수출물가·수출출하지수는 참고 |
| §2 | 성장률. 31년 명목·단가·물량과 생산지수 (표 3) |
| §2b | 달력 설명변수. 조업일수와 설·추석 세 창의 Δ12 (논문 III.2) |
| §3 | 동조성과 선후행. 달력 효과를 뺀 물량·명목 대 생산지수 Δ12 상관, 이동 상관, 교차상관 (표 4, 그림 2) |
| §4b | 공표 시차와 브리지 회귀. t+1월 1일·중순 정보 집합의 표본외 성적 (표 5). |
| §5 | 그림 1~3 |
| §6 | 검증. 논문 『관세청 수출 통계는 같은 달의 산업생산을 얼마나 먼저 알려 주는가』의 인용 수치 |

읽는 것: DB(읽기전용), 품목군 대응표 `outputs/품목군_대응표_수출.csv`, 수출물가 사본 `cache/fact_xmpi.parquet`·`dim_xmpi.parquet`.
다른 노트북의 산출물은 읽지 않는다(2026-09-14 독립화). §2b 끝의 대조 셀만 옛 캐시가 있으면 월 금액·중량을 맞대 본다.

실행: `jupyter nbconvert --to notebook --execute --inplace trend-vs-industrial-production.ipynb --ExecutePreprocessor.kernel_name=kcsdb`

## §0. 설정

In [1]:
import os, warnings
import numpy as np, pandas as pd, duckdb
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")):
    up = os.path.dirname(ROOT)
    if up == ROOT: raise FileNotFoundError("KCSDB2 루트를 찾지 못했습니다")
    ROOT = up
DB   = os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")
TT = os.path.join(ROOT, "research", "trade_trend")   # 무역 트렌드 연구 폴더(하위 폴더 셋)
HERE = os.path.join(TT, "industrial-production")
CACHE = os.path.join(HERE, "cache"); OUT = os.path.join(HERE, "outputs"); IMG = os.path.join(HERE, "img")
MAP = pd.read_csv(os.path.join(OUT, "품목군_대응표_수출.csv"), dtype=str)
con = duckdb.connect(DB, read_only=True)

def yrs(a, b): return (b - a) if b < 2026 else (2025 + 7/12 - a)  # 2026 끝점은 2025.08~2026.07 합. 2023년 창과의 간격은 2.58년
EXP15 = [s for s in MAP.group15.unique() if s != "나머지"]
# 월 금액·중량: 성질별 수출 실적(153개 코드)을 품목군 대응표로 묶는다. 다른 노트북의 캐시에 기대지 않는다(2026-09-14 독립화).
_ex = con.execute("SELECT yyyymm, temper_cd, dlr, wgt FROM fact_temper WHERE imexp='수출'").df().merge(MAP[["temper_cd", "group15"]], on="temper_cd", how="left")
assert _ex.group15.notna().all(), "대응표에 없는 수출 코드가 있다"
_gm = _ex.groupby(["yyyymm", "group15"])[["dlr", "wgt"]].sum().unstack()
W, Q = _gm["dlr"].copy(), _gm["wgt"].copy()
for M_ in (W, Q):
    M_.index = M_.index.astype(int); M_.columns.name = None
    M_["총수출"] = M_[EXP15 + ["나머지"]].sum(axis=1)
YM0, YM1 = int(W.index.min()), int(W.index.max())
# 논문이 보고하는 계열: 관세청 10대 수출 품목(현행 수출 성질별 기준)과 총수출, 비반도체. 나머지 품목군 다섯은 계산만 한다.
TOP10 = ["반도체", "승용차", "석유제품", "철강제품", "선박", "자동차부품", "무선통신기기", "컴퓨터주변기기", "정밀기기", "가전제품"]
assert set(TOP10) <= set(EXP15)
ORDER = ["총수출", "비반도체"] + sorted(TOP10, key=lambda s: -W.loc[202508:202607, s].sum())   # 최근 12개월 수출액 순(표 2~5 공통)
print("계열", len(W.columns), "| 기간", YM0, "~", YM1, "| 품목군", len(EXP15))

계열 17 | 기간 199501 ~ 202607 | 품목군 15


## §1. 대응표와 지수 구축

표 A를 코드로 둔다. 생산지수는 소분류 둘을 묶을 때 국가데이터처의 올해 가중치로, 수출물가지수는 한국은행의 2020년 가중치로
가중평균한다. 상위 항목에서 하위 하나를 빼는 자리(비반도체, 화공품에서 의약품)는 $(w_A I_A - w_B I_B)/(w_A - w_B)$로 한다.
연쇄 라스파이레스 지수를 고정 가중치로 다시 묶는 것이라 근사다. 의약품 수출물가지수는 2015.12부터라 그 전의 화공품은 화학제품 전체를 쓴다.

In [2]:
# group15 → (생산지수 더할 코드, 뺄 코드, 수출물가 더할 코드, 뺄 코드, 수출출하지수 코드(있을 때만))
MAP2 = {
    "총수출":     (["0"],            [],       ["*AA"],                 [],          ["0"]),
    "비반도체": ([],               [],       [],                      [],          ["0"]),   # 아래에서 따로 만든다
    "반도체":     (["C261"],         [],       ["30911AA"],             [],          ["C261"]),
    "승용차":     (["C301"],         [],       ["312111AA"],            [],          []),
    "자동차부품":  (["C303"],         [],       ["31213AA"],             [],          []),
    "석유제품":    (["C192"],         [],       ["3041AA"],              [],          ["C19"]),
    "철강제품":    (["C24"],          [],       ["307AA"],               [],          ["C24"]),
    "선박":       (["C311"],         [],       [],                      [],          ["C31"]),
    "무선통신기기": (["C264"],         [],       ["30951AA"],             [],          ["C264"]),
    "컴퓨터주변기기": (["C263"],       [],       ["3094AA"],              [],          ["C263"]),
    "정밀기기":    (["C27"],          [],       ["3096AA"],              [],          ["C27"]),
    "가전제품":    (["C285", "C265"], [],       ["31015AA", "30952AA"],  [],          []),
    "일반기계":    (["C29"],          [],       ["311AA"],               [],          ["C29"]),
    "화공품":     (["C20", "C22"],   [],       ["305AA"],               ["3054AA"],  ["C20", "C22"]),
    "의약품":     (["C212"],         [],       ["305411AA"],            [],          ["C21"]),
    "제조장비":    (["C292"],         [],       ["31124AA"],             [],          []),
    "이차전지":    (["C282"],         [],       ["310131AA"],            [],          []),
}
assert set(EXP15) <= set(MAP2), set(EXP15) - set(MAP2)

# 생산지수: 지표별 (yyyymm × ksic)
IPraw = con.execute("SELECT yyyymm, ksic, measure, value FROM fact_ip").df()
IP = {m: d.pivot(index="yyyymm", columns="ksic", values="value") for m, d in IPraw.groupby("measure")}
KW = con.execute("SELECT ksic, wgt FROM dim_ksic").df().set_index("ksic").wgt.to_dict()
KN = con.execute("SELECT ksic, name_ko FROM dim_ksic").df().set_index("ksic").name_ko.to_dict()
# 수출물가지수 달러기준: (yyyymm × item_cd)
# 수출물가지수는 2026-09-03에 DB에서 뺐다(참고 계산에만 쓰므로). 뺄 때 떠 둔 cache/*.parquet에서 읽는다.
XP = duckdb.sql(f"SELECT yyyymm, item_cd, value FROM '{CACHE.replace(chr(92), '/')}/fact_xmpi.parquet' WHERE imexp='수출' AND basis='D'").df() \
        .pivot(index="yyyymm", columns="item_cd", values="value")
XW = duckdb.sql(f"SELECT item_cd, wgt FROM '{CACHE.replace(chr(92), '/')}/dim_xmpi.parquet' WHERE imexp='수출'").df().set_index("item_cd").wgt.to_dict()
XN = duckdb.sql(f"SELECT item_cd, name_ko FROM '{CACHE.replace(chr(92), '/')}/dim_xmpi.parquet' WHERE imexp='수출'").df().set_index("item_cd").name_ko.to_dict()

def agg_idx(P, wd, plus, minus):
    """고정 가중 합성. 뺄 항목이 결측인 달은 더할 항목만으로 둔다(의약품 2015.12 이전)."""
    if not plus: return pd.Series(np.nan, index=P.index)
    num = sum(wd[c] * P[c] for c in plus); den = sum(wd[c] for c in plus)
    if minus:
        sub = sum(wd[c] * P[c] for c in minus); dsub = sum(wd[c] for c in minus)
        out = (num - sub) / (den - dsub)
        return out.where(sub.notna(), num / den)
    return num / den

IPX = pd.DataFrame({g: agg_idx(IP["prod"],    KW, a, b) for g, (a, b, _, _, _) in MAP2.items()}).loc[YM0:YM1]
PX  = pd.DataFrame({g: agg_idx(XP, XW, c, d)           for g, (_, _, c, d, _) in MAP2.items()}).loc[YM0:YM1]
XS  = pd.DataFrame({g: agg_idx(IP["ship_exp"], KW, e, []) for g, (_, _, _, _, e) in MAP2.items()}).loc[YM0:YM1]
for df in (IPX, PX, XS): df.index = df.index.astype(int)
W["비반도체"] = W["총수출"] - W["반도체"]; Q["비반도체"] = Q["총수출"] - Q["반도체"]

# 비반도체 생산지수: 총지수에서 C261을 빼는 대신, C261을 뺀 나머지 코드(중분류 전부 + C26의 다른 소분류)를 올해
# 가중치로 묶는다. 2026년 가중치로 빼는 식 (w0·I0 − w261·I261)/(w0 − w261)은 연쇄지수에는 근사이고,
# 반도체처럼 상대 수준이 크게 움직인 항목에서는 초기 연도가 왜곡된다. 두 방식을 견줘 둔다.
DK = con.execute("SELECT ksic, level, parent FROM dim_ksic").df()
EX_ALL = [c for c in DK[DK.level == 2].ksic if c != "C26"] + ["C262", "C263", "C264", "C265"]
# 2020년 신설 중분류(C34)는 1995~2019년이 없다. 전 기간 있는 코드만 묶는다.
def full_codes(M): return [c for c in EX_ALL if c in M.columns and M[c].loc[YM0:YM1].notna().all()]
EX_SEMI_CODES = full_codes(IP["prod"])
_drop = [c for c in EX_ALL if c not in EX_SEMI_CODES]
print(f"비반도체 합성: {len(EX_SEMI_CODES)}개 코드, 뺀 코드 {_drop} 가중치 합 {sum(KW[c] for c in _drop):.1f}/10000")
IPX["비반도체"] = agg_idx(IP["prod"],    KW, EX_SEMI_CODES, []).loc[YM0:YM1].values
ipx_sub = agg_idx(IP["prod"], KW, ["0"], ["C261"]).loc[YM0:YM1]
_c = np.corrcoef(np.log(IPX["비반도체"]).diff(12).dropna(), np.log(ipx_sub).diff(12).dropna())[0, 1]
print(f"비반도체 생산지수: 합성 대 뺄셈의 Δ12 상관 {_c:.3f}, 2020 평균 {IPX['비반도체'].loc[202001:202012].mean():.1f}")

# 비반도체 수출물가지수: 가격이 있는 열세 품목군(선박 제외)의 연쇄 Törnqvist. 가중치는 연도별 수출액 비중.
# 2020년 고정 가중치로 총지수에서 반도체를 빼면 1990년대에 음수가 나온다 — 반도체 가격이 100배 넘게 떨어졌기 때문.
PRICED = [g for g in EXP15 if g not in ("반도체", "선박")]
def chain_tornqvist(groups):
    sh = W[groups].groupby(W.index // 100).sum(); sh = sh.div(sh.sum(axis=1), axis=0)
    dlp = np.log(PX[groups]).diff()
    yr = pd.Series(dlp.index // 100, index=dlp.index)
    w_cur = sh.reindex(yr.values).set_index(dlp.index); w_prev = sh.reindex(yr.values - 1).set_index(dlp.index)
    w_prev = w_prev.fillna(w_cur)
    wbar = 0.5 * (w_cur + w_prev)
    ok = dlp.notna()
    step = (wbar.where(ok) * dlp).sum(axis=1) / wbar.where(ok).sum(axis=1)   # 결측 품목(의약품·제조장비 초기)은 그 달 가중치에서 뺀다
    lp = step.fillna(0).cumsum()
    idx = np.exp(lp); return idx / idx.loc[202001:202012].mean() * 100
PX["비반도체"] = chain_tornqvist(PRICED)
px_all = chain_tornqvist(PRICED + ["반도체"])
_c = np.corrcoef(np.log(px_all).diff(12).dropna(), np.log(PX["총수출"]).diff(12).dropna())[0, 1]
print(f"열네 품목군 연쇄 물가지수 대 한국은행 총지수: Δ12 상관 {_c:.3f}, 31년 연평균 {np.log(px_all.iloc[-1]/px_all.iloc[0])*100/yrs(1995,2026):.2f}% 대 "
      f"{np.log(PX['총수출'].iloc[-1]/PX['총수출'].iloc[0])*100/yrs(1995,2026):.2f}%")
XS["비반도체"] = agg_idx(IP["ship_exp"], KW, ["0"], ["C261"]).loc[YM0:YM1].values   # 수출출하는 소분류가 C26뿐이라 뺄셈으로
R = (W[list(MAP2)] / PX * 100)                     # 실질 수출 (2020년 달러)

# 단가와 물량은 품목군 중량을 그대로 쓰지 않고 첫 갈래와 같은 코드 단위 Törnqvist로 가른다.
# 품목군 중량은 값이 작고 무거운 코드에 끌려간다. 반도체 중량은 2024년에 한 해 만에 3분의 1이 됐는데,
# 금액 비중 1%인 기타 개별소자(44307)의 중량이 186천 톤에서 41천 톤으로 준 것이고 메모리는 5.5천 톤 그대로다.
raw = con.execute("SELECT yyyymm, temper_cd, dlr, wgt FROM fact_temper WHERE imexp='수출'").df().merge(MAP[["temper_cd", "group15"]], on="temper_cd")
GROUP_CODES = {g: MAP.loc[MAP.group15 == g, "temper_cd"].tolist() for g in EXP15}
GROUP_CODES["총수출"] = MAP.temper_cd.tolist(); GROUP_CODES["비반도체"] = MAP.loc[MAP.group15 != "반도체", "temper_cd"].tolist()
def tornqvist(d0, d1):
    """d0·d1: temper_cd 인덱스, 열 dlr·wgt. 반환 (Δln V, Δln P, Δln Q), 로그 차이 그대로."""
    both = d0.join(d1, lsuffix="0", rsuffix="1", how="inner")
    both = both[(both.dlr0 > 0) & (both.dlr1 > 0) & (both.wgt0 > 0) & (both.wgt1 > 0)]
    w = 0.5 * (both.dlr0 / both.dlr0.sum() + both.dlr1 / both.dlr1.sum())
    dlp = (w * np.log((both.dlr1 / both.wgt1) / (both.dlr0 / both.wgt0))).sum()
    dlv = np.log(d1.dlr.sum() / d0.dlr.sum())
    return dlv, dlp, dlv - dlp
# 월 연쇄 물량지수 (2020=100): 달마다 앞 달과의 Törnqvist 단가 변화를 빼서 잇는다
MV = raw.pivot(index="yyyymm", columns="temper_cd", values="dlr").fillna(0); MW = raw.pivot(index="yyyymm", columns="temper_cd", values="wgt").fillna(0)
def chain_q(codes):
    v, w = MV[codes], MW[codes]
    ok = (v > 0) & (w > 0) & (v.shift(1) > 0) & (w.shift(1) > 0)
    sh = v.div(v.sum(axis=1), axis=0); wbar = 0.5 * (sh + sh.shift(1))
    dlp_i = np.log((v / w) / (v.shift(1) / w.shift(1)))
    dlp = (wbar.where(ok) * dlp_i.where(ok)).sum(axis=1) / wbar.where(ok).sum(axis=1)
    dlv = np.log(v.sum(axis=1) / v.sum(axis=1).shift(1))
    lq = (dlv - dlp).fillna(0).cumsum(); q = np.exp(lq)
    return q / q.loc[202001:202012].mean() * 100
QT = pd.DataFrame({g: chain_q(c) for g, c in GROUP_CODES.items()}).loc[YM0:YM1]
print("월 연쇄 Törnqvist 물량지수 31년 연평균(%):", {g: round(np.log(QT[g].loc[202508:202607].mean() / QT[g].loc[199501:199512].mean()) * 100 / yrs(1995, 2026), 1) for g in ["총수출", "비반도체", "반도체"]})

rows = []
for g, (a, b, c, d, e) in MAP2.items():
    if g == "비반도체":
        rows.append({"품목군": g, "생산지수": "C26을 뺀 모든 중분류와 C262~C265의 가중 합성", "수출물가지수": "열세 품목군의 연쇄 Törnqvist(수출액 가중)",
                     "수출출하지수": "0 −C261", "실질 첫 달": int(R[g].first_valid_index())}); continue
    rows.append({"품목군": g, "생산지수": "+".join(f"{k} {KN[k]}" for k in a) + ("".join(f" −{k} {KN[k]}" for k in b)),
                 "수출물가지수": "+".join(f"{k} {XN[k]}" for k in c) + ("".join(f" −{k} {XN[k]}" for k in d)) if c else "없음",
                 "수출출하지수": "+".join(e) if e else "없음",
                 "실질 첫 달": int(R[g].first_valid_index()) if R[g].notna().any() else -1})
TAB_MAP = pd.DataFrame(rows).set_index("품목군")
TAB_MAP.to_csv(os.path.join(OUT, "품목군_대응표_산업물가.csv"), encoding="utf-8-sig")
print(TAB_MAP.to_string())

비반도체 합성: 31개 코드, 뺀 코드 ['C34'] 가중치 합 85.8/10000
비반도체 생산지수: 합성 대 뺄셈의 Δ12 상관 0.929, 2020 평균 100.0
열네 품목군 연쇄 물가지수 대 한국은행 총지수: Δ12 상관 0.950, 31년 연평균 -2.70% 대 -0.76%


월 연쇄 Törnqvist 물량지수 31년 연평균(%): {'총수출': np.float64(3.0), '비반도체': np.float64(3.1), '반도체': np.float64(1.0)}
                                                     생산지수                           수출물가지수   수출출하지수  실질 첫 달
품목군                                                                                                        
총수출                                                 0 총지수                          *AA 총지수        0  199501
비반도체                      C26을 뺀 모든 중분류와 C262~C265의 가중 합성     열세 품목군의 연쇄 Törnqvist(수출액 가중)  0 −C261  199501
반도체                                          C261 반도체 제조업                      30911AA 반도체     C261  199501
승용차                                C301 자동차용 엔진 및 자동차 제조업                     312111AA 승용차       없음  199501
자동차부품                                  C303 자동차 신품 부품 제조업                    31213AA 자동차부품       없음  199501
석유제품                                      C192 석유 정제품 제조업                   3041AA 석탄및석유제품      C19  199501
철강제품                          

## §2. 명목·단가·물량과 생산 성장률의 비교

1995년과 마지막 12개월의 연간값을 끝점으로 연평균 로그 변화를 낸다. 금액·중량은 연간 합, 지수는 연간 평균이다.
2026은 2025.08~2026.07의 12개월이다. 명목 $\Delta \ln V$, 코드 단위 Törnqvist 단가 $\Delta \ln U$, 물량 $\Delta \ln Q = \Delta \ln V - \Delta \ln U$(논문 III.1)와
생산지수의 연평균 성장률을 나란히 둔다(논문 표 3). 수출물가 $\Delta \ln P$, 실질 $\Delta \ln V - \Delta \ln P$, 단가−물가, 수출출하지수는 참고 열이다.

In [3]:
def annual(M, how):
    """월 표를 연 표로. 2026은 2025.08~2026.07."""
    y = M.loc[:202512].groupby(M.loc[:202512].index // 100).agg(how)
    ttm = M.loc[202508:202607].agg(how).rename(2026)
    return pd.concat([y, ttm.to_frame().T])
Vy = annual(W[list(MAP2)], "sum")
Py, IPy, XSy = annual(PX, "mean"), annual(IPX, "mean"), annual(XS, "mean")
Ry = Vy / Py * 100
# 코드 단위 연간 금액·중량 (끝점용). 2026은 2025.08~2026.07의 12개월 합.
ann = raw[raw.yyyymm <= 202512].assign(year=lambda d: d.yyyymm // 100).groupby(["year", "temper_cd"])[["dlr", "wgt"]].sum().reset_index()
ttm = raw[(raw.yyyymm >= 202508) & (raw.yyyymm <= 202607)].groupby("temper_cd")[["dlr", "wgt"]].sum().reset_index().assign(year=2026)
A = pd.concat([ann, ttm], ignore_index=True).set_index(["year", "temper_cd"])
def tq(g, y0, y1):
    d0 = A.loc[y0].reindex(GROUP_CODES[g]).dropna(); d1 = A.loc[y1].reindex(GROUP_CODES[g]).dropna()
    return tornqvist(d0[["dlr", "wgt"]], d1[["dlr", "wgt"]])

In [4]:
# 31년 전체(1995→2026.07): 실질 수출 대 생산지수 — 어느 품목군이 생산보다 빨리 자랐나
n_all = yrs(1995, 2026)
full = pd.DataFrame({
    "명목": np.log(Vy.loc[2026] / Vy.loc[1995]) * 100 / n_all,
    "물가": np.log(Py.loc[2026] / Py.loc[1995]) * 100 / n_all,
    "단가": pd.Series({g: tq(g, 1995, 2026)[1] * 100 / n_all for g in MAP2}),
    "물량": pd.Series({g: tq(g, 1995, 2026)[2] * 100 / n_all for g in MAP2}),
    "생산지수": np.log(IPy.loc[2026] / IPy.loc[1995]) * 100 / n_all,
    "수출출하지수": np.log(XSy.loc[2026] / XSy.loc[1995]) * 100 / n_all})
full["실질"] = full["명목"] - full["물가"]
full["단가−물가"] = full["단가"] - full["물가"]; full["실질−생산"] = full["실질"] - full["생산지수"]; full["물량−생산"] = full["물량"] - full["생산지수"]
TAB_FULL = full.reindex(ORDER)[["명목", "물가", "실질", "단가", "물량", "단가−물가", "생산지수", "실질−생산", "물량−생산", "수출출하지수"]]
TAB_FULL.to_csv(os.path.join(OUT, "tab2_full_growth.csv"), encoding="utf-8-sig")
print(TAB_FULL.round(1).to_string())

           명목    물가    실질    단가   물량  단가−물가  생산지수  실질−생산  물량−생산  수출출하지수
총수출       6.5  -1.4   7.9   4.5  2.0    5.9   4.4    3.4   -2.5     7.3
비반도체      5.6  -0.8   6.4   3.0  2.5    3.9   1.9    4.5    0.6     6.2
반도체       9.5 -15.3  24.8   9.2  0.3   24.5  19.8    5.0  -19.5    20.3
승용차       7.7   0.5   7.1   1.7  5.9    1.2   2.3    4.9    3.7     NaN
석유제품     10.2   5.5   4.7   5.1  5.1   -0.5   2.1    2.6    3.1     4.9
철강제품      5.1   2.3   2.8   1.7  3.5   -0.7   1.5    1.3    2.0     3.9
컴퓨터주변기기   6.7  -7.8  14.5  10.9 -4.2   18.6   2.9   11.6   -7.0     4.4
선박        5.9   NaN   NaN   7.3 -1.4    NaN   4.8    NaN   -6.2     5.1
무선통신기기    9.7  -8.4  18.1   8.5  1.2   16.9   3.8   14.2   -2.7     6.4
자동차부품    10.0   0.3   9.6   1.0  9.0    0.6   5.1    4.5    3.9     NaN
정밀기기      8.7  -2.2  10.9   2.5  6.2    4.7   2.9    8.0    3.2     4.0
가전제품     -0.5  -2.8   2.3   2.3 -2.9    5.1  -0.8    3.0   -2.1     NaN


## §2b. 달력 설명변수 (논문 III.2)

원계열의 전년 동월 대비에도 두 해의 조업일수 차이와 설·추석의 달 이동은 남는다. 그 달의 달력은 미리 알 수 있으므로
조업일수(월~금, 공휴일 제외)의 로그와 설·추석 세 창(연휴 앞 20일, 연휴 사흘, 연휴 뒤 10일에 든 날의 몫)을 설명변수로 만든다.
달력월 평균으로 중심화하고 Δ12를 취한다. 동조성은 수출과 생산 각각에서 이 일곱 변수로 설명되는 부분을 회귀로 뺀 잔차끼리 재고,
브리지 회귀는 모든 모형에 이 일곱 변수를 넣는다. 끝 셀은 옛 캐시(기술분석 노트북 산출물)가 있으면 월 금액·중량을 맞대 보기만 한다.

In [5]:
import datetime as dt
from korean_lunar_calendar import KoreanLunarCalendar
LEAD = 12
YM_LEAD = (YM1 // 100 + (YM1 % 100 + LEAD - 1) // 12) * 100 + (YM1 % 100 + LEAD - 1) % 12 + 1
CAL_VARS = ["ln_wd", "seol_pre", "seol_dur", "seol_post", "chu_pre", "chu_dur", "chu_post"]
wd_ = con.execute("SELECT base_ym AS ym, SUM(workdays) AS workdays FROM dim_workday10d GROUP BY 1 ORDER BY 1").df().set_index("ym")
def _lunar(y, m, d):
    k = KoreanLunarCalendar(); k.setLunarDate(y, m, d, False); return dt.date.fromisoformat(k.SolarIsoFormat())
def _hol_reg(anchor, w_pre, w_post, name):
    rows = []
    for y, d0 in anchor.items():
        dur = [d0 + dt.timedelta(k) for k in (-1, 0, 1)]
        pre = [dur[0] - dt.timedelta(k) for k in range(1, w_pre + 1)]; post = [dur[-1] + dt.timedelta(k) for k in range(1, w_post + 1)]
        for kk, win in [("pre", pre), ("dur", dur), ("post", post)]:
            for d in win: rows.append({"ym": d.year * 100 + d.month, "var": f"{name}_{kk}", "w": 1 / len(win)})
    return pd.DataFrame(rows).groupby(["ym", "var"])["w"].sum().unstack(fill_value=0.0)
_H = _hol_reg({y: _lunar(y, 1, 1) for y in range(1994, 2028)}, 20, 10, "seol").add(_hol_reg({y: _lunar(y, 8, 15) for y in range(1994, 2028)}, 20, 10, "chu"), fill_value=0.0)
def cal_regressors(start):
    """달력 설명변수를 start~YM_LEAD로 자르고 start~YM1의 달력월 평균으로 중심화한다."""
    X_ = wd_.join(_H).fillna(0.0); X_["ln_wd"] = np.log(X_["workdays"]); X_ = X_.loc[start:YM_LEAD].copy(); X_["month"] = X_.index % 100
    for c in CAL_VARS:
        mm = X_.loc[:YM1].groupby("month")[c].mean(); X_[c] = X_[c] - X_["month"].map(mm)
    return X_[CAL_VARS]
Xcal_exp = cal_regressors(YM0)
dCAL = Xcal_exp.diff(12)
print("달력 설명변수:", CAL_VARS, "| Δ12 범위", dCAL.dropna().index.min(), "~", dCAL.dropna().index.max())

달력 설명변수: ['ln_wd', 'seol_pre', 'seol_dur', 'seol_post', 'chu_pre', 'chu_dur', 'chu_post'] | Δ12 범위 199601 ~ 202707


In [6]:
# 옛 캐시와의 대조(출력만): 독립화 전에는 기술분석 노트북의 캐시를 읽었다. 파일이 있으면 맞대 보고, 차이가 있으면 원인을 볼 것.
def _cmp(label, new, path, col):
    if not os.path.exists(path): print(f"{label}: 옛 캐시 없음, 건너뜀"); return
    old = pd.read_csv(path).replace({"series": {"반도체 제외": "비반도체"}}).pivot(index="yyyymm", columns="series", values=col); old.index = old.index.astype(int)   # 옛 이름으로 만든 파일도 읽는다
    cols = [c for c in new.columns if c in old.columns]
    print(f"{label}: 맞댄 계열 {len(cols)} | 최대 절대 차이 {(new[cols] - old[cols].reindex(new.index)).abs().max().max():.3e}")
_cmp("월 금액", W[["총수출"] + EXP15], os.path.join(TT, "descriptive", "cache", "series_monthly.csv"), "dlr")
_cmp("월 중량", Q[["총수출"] + EXP15], os.path.join(TT, "descriptive", "cache", "series_monthly.csv"), "wgt")

월 금액: 맞댄 계열 16 | 최대 절대 차이 0.000e+00
월 중량: 맞댄 계열 16 | 최대 절대 차이 0.000e+00


## §3. 동조성과 선후행

전년 동월 대비 로그 차이 $\Delta_{12}$에서 달력 설명변수로 설명되는 부분을 전체 기간 회귀로 뺀 잔차의 상관. 물량지수의 $\Delta_{12}$는 $t$월과 $t-12$월의 양자간 Törnqvist로
직접 구해 월 연쇄를 거치지 않는다(논문 III.1). 물량과 명목 수출을 생산지수(원지수)와 맞댄다(논문 표 4). 시기별 변화는 물량 대 생산지수의 60개월 이동 상관으로 본다(그림 2).
선후행은 같은 잔차를 한 번 차분한 계열의 교차상관으로 본다 — 전년 동월 대비는 이웃한 달끼리 11개월이 겹쳐 교차상관이 평탄하기 때문이다.
실질 수출(수출물가로 나눈 것)과 수출출하지수의 상관은 참고 열이다.

In [7]:
def d12(M): return np.log(M).diff(12)
# 물량지수의 Δ12: t월과 t-12월 사이의 양자간 Törnqvist(연쇄를 거치지 않는다). 두 달 모두 금액·중량이 있는 코드로 단가 변화를 가중한다.
def bilat_q(codes):
    v, w = MV[codes], MW[codes]; v0, w0 = v.shift(12), w.shift(12)
    ok = (v > 0) & (w > 0) & (v0 > 0) & (w0 > 0)
    s1 = v.where(ok); s1 = s1.div(s1.sum(axis=1), axis=0); s0 = v0.where(ok); s0 = s0.div(s0.sum(axis=1), axis=0)
    dlp = (0.5 * (s0 + s1) * np.log((v / w) / (v0 / w0)).where(ok)).sum(axis=1, min_count=1)
    return (np.log(v.sum(axis=1) / v0.sum(axis=1)) - dlp).replace([np.inf, -np.inf], np.nan)
dQ_ = pd.DataFrame({g: bilat_q(GROUP_CODES[g]) for g in MAP2}); dQ_.index = dQ_.index.astype(int); dQ_ = dQ_.loc[YM0:YM1]
dQchain = d12(QT.reindex(columns=list(MAP2)))      # 월 연쇄 지수의 Δ12(대조용)
_j = pd.concat([dQ_["총수출"], dQchain["총수출"]], axis=1).dropna()
print(f"물량 Δ12 양자간 대 월 연쇄(총수출): 상관 {_j.corr().iloc[0, 1]:.3f}, 평균 차이 {(_j.iloc[:, 1] - _j.iloc[:, 0]).mean() * 100:.2f}%p")
dR, dV, dIP, dXS = d12(R), d12(W[list(MAP2)]), d12(IPX), d12(XS)
def decal(M):
    """각 열의 Δ12에서 달력 설명변수로 설명되는 부분을 전체 기간 OLS로 뺀 잔차."""
    out = {}
    for c in M.columns:
        d = pd.DataFrame({"y": M[c]}).join(dCAL).dropna()
        if len(d) < 36: out[c] = M[c] * np.nan; continue
        out[c] = sm.OLS(d.y, sm.add_constant(d[CAL_VARS])).fit().resid.reindex(M.index)
    return pd.DataFrame(out)
eR, eQ, eV, eIP, eXS = decal(dR), decal(dQ_), decal(dV), decal(dIP), decal(dXS)
CAL_R2 = pd.DataFrame({lab: 1 - E.var() / D.where(E.notna()).var() for lab, E, D in [("물량", eQ, dQ_), ("명목", eV, dV), ("생산", eIP, dIP)]})
print("달력이 설명하는 Δ12 분산 몫:"); print(CAL_R2.loc[["총수출", "비반도체", "반도체"]].round(3).to_string())
# 단가(명목 − 물량)의 Δ12 가운데 달력이 설명하는 몫: 물량과 명목의 달력 몫 차이가 단가의 달력 반응 때문인지 본다
dU_ = dV - dQ_; eU_ = decal(dU_)
CAL_R2_U = 1 - eU_.var() / dU_.where(eU_.notna()).var()
print("단가 Δ12의 달력 몫(총수출):", round(CAL_R2_U["총수출"], 3))
rows = []
for g in MAP2:
    r = {"계열": g}
    def cc(x, y):
        j = pd.concat([x, y], axis=1).dropna()
        return (j.iloc[:, 0].corr(j.iloc[:, 1]), len(j)) if len(j) > 24 else (np.nan, len(j))
    r["물량·생산"], r["n"] = cc(eQ[g], eIP[g])
    r["명목·생산"], _ = cc(eV[g], eIP[g])
    r["실질·생산(참고)"], _ = cc(eR[g], eIP[g])
    r["물량·수출출하(참고)"], _ = cc(eQ[g], eXS[g])
    rows.append(r)
TAB_SYNC2 = pd.DataFrame(rows).set_index("계열").reindex(ORDER)
TAB_SYNC2.to_csv(os.path.join(OUT, "tab2_sync.csv"), encoding="utf-8-sig")
print(TAB_SYNC2.round(2).to_string())

물량 Δ12 양자간 대 월 연쇄(총수출): 상관 0.984, 평균 차이 0.06%p


달력이 설명하는 Δ12 분산 몫:


         물량     명목     생산
총수출   0.208  0.036  0.126
비반도체  0.239  0.061  0.219
반도체   0.017  0.010  0.005
단가 Δ12의 달력 몫(총수출): 0.024
         물량·생산    n  명목·생산  실질·생산(참고)  물량·수출출하(참고)
계열                                                
총수출       0.58  367   0.63       0.61         0.74
비반도체      0.47  367   0.64       0.42         0.75
반도체       0.38  367   0.51       0.53         0.38
승용차       0.66  367   0.72       0.70          NaN
석유제품      0.56  367   0.40       0.47         0.81
철강제품      0.19  367   0.41       0.02         0.68
컴퓨터주변기기   0.45  367   0.27       0.36         0.43
선박        0.18  367   0.18        NaN         0.15
무선통신기기    0.45  367   0.48       0.49         0.55
자동차부품     0.54  367   0.56       0.49          NaN
정밀기기      0.11  367   0.18       0.15         0.18
가전제품      0.42  367   0.46       0.48          NaN


In [8]:
# 선후행: 달력을 뺀 Δ12 잔차를 한 번 차분한 계열의 교차상관 corr(x_{t-k}, ip_t), k=-12..12. k>0이면 수출이 앞선다.
# 차분하지 않은 Δ12 잔차는 이웃한 달끼리 11개월이 겹쳐 교차상관이 평탄하다. 대역은 ±2/√n.
def xcorr(x, y, K=12):
    out = {}
    for k in range(-K, K + 1):
        j = pd.concat([x.shift(k), y], axis=1).dropna()
        out[k] = j.iloc[:, 0].corr(j.iloc[:, 1]) if len(j) > 24 else np.nan
    return pd.Series(out)
rows = []
for g in MAP2:
    xq, xv = xcorr(eQ[g].diff(), eIP[g].diff()), xcorr(eV[g].diff(), eIP[g].diff())
    n_ = pd.concat([eQ[g].diff(), eIP[g].diff()], axis=1).dropna().shape[0]; band = 2 / np.sqrt(n_)
    oq, ov = xq.drop(xq.idxmax()), xv.drop(xv.idxmax())
    rows.append({"계열": g, "물량 최대 시차": int(xq.idxmax()), "물량 최대 상관": xq.max(), "물량 동시 상관": xq[0],
                 "명목 최대 시차": int(xv.idxmax()), "명목 최대 상관": xv.max(), "명목 동시 상관": xv[0],
                 "대역": band, "물량 대역 밖 다른 시차 수": int((oq.abs() > band).sum()), "명목 대역 밖 다른 시차 수": int((ov.abs() > band).sum()),
                 "물량 다른 시차 최대": oq.abs().max(), "명목 다른 시차 최대": ov.abs().max()})
TAB_XC = pd.DataFrame(rows).set_index("계열").reindex(ORDER)
TAB_XC.to_csv(os.path.join(OUT, "tab2_xcorr_d12.csv"), encoding="utf-8-sig")
print(TAB_XC.round(2).to_string())
XC = pd.DataFrame({g: xcorr(eQ[g].diff(), eIP[g].diff()) for g in ["총수출", "비반도체", "반도체"]})
print(XC.round(2).T.to_string())

         물량 최대 시차  물량 최대 상관  물량 동시 상관  명목 최대 시차  명목 최대 상관  명목 동시 상관   대역  물량 대역 밖 다른 시차 수  명목 대역 밖 다른 시차 수  물량 다른 시차 최대  명목 다른 시차 최대
계열                                                                                                                                  
총수출             0      0.41      0.41         0      0.51      0.51  0.1                5                6         0.17         0.21
비반도체            0      0.42      0.42         0      0.46      0.46  0.1                3                6         0.14         0.17
반도체             0      0.17      0.17         0      0.52      0.52  0.1                3                4         0.13         0.31
승용차             0      0.77      0.77         0      0.80      0.80  0.1                6                8         0.41         0.42
석유제품            0      0.23      0.23         0      0.22      0.22  0.1                4                2         0.15         0.13
철강제품            0      0.18      0.18         0      0.24      0.24  

## §4b. 공표 시차와 브리지 회귀

$t$월 자료의 공표는 월간 수출입 동향 $t+1$월 1일, 관세청 확정치(중량 포함) $t+1$월 중순, 광공업생산지수 $t+1$월 마지막 평일이다.
두 예측 시점 모두 $t-1$월 생산까지 공표되어 있다. **기준 모형**은 전월·전전월 생산의 $\Delta_{12}$, 기저효과 $\Delta_1 \ln Y_{t-12}$,
달력 7개를 쓴다(항등식 $\Delta_{12}\ln Y_t = \Delta_{12}\ln Y_{t-1} + \Delta_1\ln Y_t - \Delta_1\ln Y_{t-12}$의 마지막 항은 미리 알려져 있다).
**1일 모형**은 여기에 $t$월 명목 수출을, **중순 모형**은 명목 수출과 물량지수를 함께 더한다(정보가 시점마다 쌓이도록 내포).
확장창 표본외 2010.01~, 오차 비교는 Clark–West(2007) 검정(HAC 12), 계수의 $t$값은 Newey–West(12) 표준오차로도 낸다(논문 표 5).
하위 기간과 코로나 기간 제외(논문 표 6), 제조업지수·달력 구간·합산 예측·총지수 재구성은 강건성 확인이다.

In [9]:
lnIP = np.log(IPX)
def base_X(g, dcal=None, lnY=None):
    # 기준 모형 설명변수: 전월·전전월 생산 Δ12, 기저효과 Δ1 ln Y_{t-12}, 달력 7개 — 모두 t+1월 1일에 알려져 있다
    y = d12(np.exp(lnY)) if lnY is not None else dIP[g]; L = lnY if lnY is not None else lnIP[g]
    X = pd.DataFrame({"yl1": y.shift(1), "yl2": y.shift(2), "base": L.diff().shift(12)}).join(dCAL if dcal is None else dcal)
    return y, X
def oos(y, X, start=201001):
    d = pd.concat([y.rename("y"), X], axis=1).dropna(); cols = list(X.columns); P = []
    for t in d.index[d.index >= start]:
        tr = d.loc[:t].iloc[:-1]; m = sm.OLS(tr.y, sm.add_constant(tr[cols])).fit()
        P.append((t, d.loc[t, "y"], float(m.params.values @ np.r_[1.0, d.loc[t, cols].values.astype(float)])))
    return pd.DataFrame(P, columns=["t", "y", "f"]).set_index("t")
def rmse(o, a=None, b=None, drop=None):
    s_ = o.loc[a:b] if a else o
    if drop: s_ = s_.drop(s_.loc[drop[0]:drop[1]].index)
    return np.sqrt(((s_.y - s_.f) ** 2).mean()) * 100
def cw_t(o0, o1):
    # Clark-West(2007): 내포 모형의 MSPE 조정 차이 f = e0² − (e1² − (f0−f1)²). 단측, HAC(12) t.
    j_ = o0.join(o1, rsuffix="1"); f = (j_.y - j_.f) ** 2 - ((j_.y1 - j_.f1) ** 2 - (j_.f - j_.f1) ** 2)
    return sm.OLS(f.values, np.ones(len(f))).fit(cov_type="HAC", cov_kwds={"maxlags": 12}).tvalues[0]
def fit_full(y, X):
    d = pd.concat([y.rename("y"), X], axis=1).dropna(); Z = sm.add_constant(d[list(X.columns)])
    return sm.OLS(d.y, Z).fit(), sm.OLS(d.y, Z).fit(cov_type="HAC", cov_kwds={"maxlags": 12}), len(d)
def chg(a, b): return (b / a - 1) * 100

_tot = W.loc[202508:202607, "총수출"].sum()
rows, OOS = [], {}
for g in ORDER:
    y, X0 = base_X(g); X1 = X0.assign(v=dV[g]); X2 = X1.assign(q=dQ_[g])
    o0, o1, o2 = oos(y, X0), oos(y, X1), oos(y, X2); OOS[g] = (o0, o1, o2)
    f0, _, n = fit_full(y, X0); f1, h1, _ = fit_full(y, X1); f2, h2, _ = fit_full(y, X2)
    rows.append({"계열": g, "수출 비중": W.loc[202508:202607, g].sum() / _tot * 100, "n": n, "표본외 n": len(o0),
                 "기준 R2": f0.rsquared, "기준 RMSE": rmse(o0),
                 "명목 계수": f1.params["v"], "명목 t": f1.tvalues["v"], "명목 t HAC": h1.tvalues["v"], "1일 RMSE": rmse(o1), "1일 변화": chg(rmse(o0), rmse(o1)), "1일 CW": cw_t(o0, o1),
                 "중순 물량 계수": f2.params["q"], "중순 물량 t HAC": h2.tvalues["q"], "중순 RMSE": rmse(o2), "중순 변화": chg(rmse(o0), rmse(o2)),
                 "중순 CW(기준 대비)": cw_t(o0, o2), "중순 CW(1일 대비)": cw_t(o1, o2)})
TAB_ITEM = pd.DataFrame(rows).set_index("계열")
TAB_ITEM.to_csv(os.path.join(OUT, "tab_bridge_items.csv"), encoding="utf-8-sig")
print(TAB_ITEM.round(2).to_string())

# 전월 수출(t+1월 1일에 이미 확정치가 있다): 생산 자신의 과거를 넘는 선행 정보가 있는가
rows = []
for g in ["총수출", "비반도체", "반도체"]:
    y, X0 = base_X(g); o0 = OOS[g][0]
    for lab, x in [("명목", dV[g].shift(1)), ("물량", dQ_[g].shift(1))]:
        o = oos(y, X0.assign(x=x)); rows.append({"계열": g, "척도": lab, "RMSE": rmse(o), "변화": chg(rmse(o0), rmse(o)), "CW": cw_t(o0, o)})
TAB_LAG = pd.DataFrame(rows); print(TAB_LAG.round(2).to_string(index=False))

# 하위 기간과 코로나 기간(2020.03~12) 제외 (논문 표 6)
PER = [("2010~2019", 201001, 201912, None), ("2020~2021", 202001, 202112, None), ("2022~2026.07", 202201, 202607, None), ("코로나 제외", 201001, 202607, (202003, 202012))]
rows = []
for g in ["총수출", "비반도체", "반도체"]:
    o0, o1, o2 = OOS[g]
    for lab, a, b, dr in PER:
        rows.append({"계열": g, "기간": lab, "기준 RMSE": rmse(o0, a, b, dr), "1일 변화": chg(rmse(o0, a, b, dr), rmse(o1, a, b, dr)),
                     "중순 변화": chg(rmse(o0, a, b, dr), rmse(o2, a, b, dr)), "개월": len((o0.loc[a:b]).drop(o0.loc[dr[0]:dr[1]].index) if dr else o0.loc[a:b])})
TAB_SUB = pd.DataFrame(rows); TAB_SUB.to_csv(os.path.join(OUT, "tab_bridge_subperiods.csv"), index=False, encoding="utf-8-sig")
print(TAB_SUB.round(1).to_string(index=False))

# 강건성 1: 총수출 대 제조업지수(C) — 총지수에는 전기·가스업이 들어 있다
ROB = {}
lnC = np.log(IP["prod"]["C"]); lnC.index = lnC.index.astype(int); lnC = lnC.loc[YM0:YM1]
yC, XC0 = base_X(None, lnY=lnC); oC0, oC1, oC2 = oos(yC, XC0), oos(yC, XC0.assign(v=dV["총수출"])), oos(yC, XC0.assign(v=dV["총수출"], q=dQ_["총수출"]))
ROB["제조업 1일"], ROB["제조업 중순"] = chg(rmse(oC0), rmse(oC1)), chg(rmse(oC0), rmse(oC2))
ROB["제조업 1일 CW"], ROB["제조업 중순 CW"] = cw_t(oC0, oC1), cw_t(oC0, oC2)
# 강건성 2: 설·추석 구간 (앞, 뒤) 일수를 바꾼 달력
def dcal_with(pre, post):
    H = _hol_reg({yy: _lunar(yy, 1, 1) for yy in range(1994, 2028)}, pre, post, "seol").add(_hol_reg({yy: _lunar(yy, 8, 15) for yy in range(1994, 2028)}, pre, post, "chu"), fill_value=0.0)
    X_ = wd_.join(H).fillna(0.0); X_["ln_wd"] = np.log(X_["workdays"]); X_ = X_.loc[YM0:YM_LEAD].copy(); X_["month"] = X_.index % 100
    for c in CAL_VARS:
        if c not in X_: X_[c] = 0.0
        mm = X_.loc[:YM1].groupby("month")[c].mean(); X_[c] = X_[c] - X_["month"].map(mm)
    return X_[CAL_VARS].diff(12)
CALW = [(10, 5), (20, 10), (30, 15), (20, 20)]
rows = []
for pre, post in CALW:
    dc = dcal_with(pre, post)
    for g in ["총수출", "비반도체"]:
        y, X0 = base_X(g, dcal=dc); a0, a1, a2 = oos(y, X0), oos(y, X0.assign(v=dV[g])), oos(y, X0.assign(v=dV[g], q=dQ_[g]))
        rows.append({"구간": f"{pre},{post}", "계열": g, "1일 변화": chg(rmse(a0), rmse(a1)), "중순 변화": chg(rmse(a0), rmse(a2))})
TAB_CALW = pd.DataFrame(rows); print(TAB_CALW.round(1).to_string(index=False))
# 강건성 3: 반도체와 비반도체를 나눠 예측해 합친 총지수 대 직접 예측(2026년 가중치)
w_s = KW["C261"] / KW["0"]
for k, lab in [(1, "1일"), (2, "중순")]:
    comb = (1 - w_s) * OOS["비반도체"][k].f + w_s * OOS["반도체"][k].f
    jj = pd.DataFrame({"y": dIP["총수출"], "c": comb}).dropna()
    ROB[f"합산 {lab} RMSE"] = np.sqrt(((jj.y - jj.c) ** 2).mean()) * 100; ROB[f"직접 {lab} RMSE"] = rmse(OOS["총수출"][k])
ROB["반도체 가중"] = w_s
# 강건성 4: 같은 고정 가중 방식으로 총지수를 재구성해 공식 총지수와 비교(비반도체 합성의 정확도)
_full2 = [c for c in DK[DK.level == 2].ksic if c in IP["prod"].columns and IP["prod"][c].loc[YM0:YM1].notna().all()]
_rec = agg_idx(IP["prod"], KW, _full2, []).loc[YM0:YM1]; _rec.index = _rec.index.astype(int)
_jr = pd.concat([d12(_rec), dIP["총수출"]], axis=1).dropna(); _er = _jr.iloc[:, 0] - _jr.iloc[:, 1]
ROB["재구성 상관"] = _jr.corr().iloc[0, 1]
for a, b in [(199601, 200512), (200601, 201512), (201601, 202607), (201001, 202607)]: ROB[f"재구성 RMSE {a // 100}~{b // 100}"] = np.sqrt((_er.loc[a:b] ** 2).mean()) * 100
print({k: round(v, 3) for k, v in ROB.items()})

          수출 비중    n  표본외 n  기준 R2  기준 RMSE  명목 계수   명목 t  명목 t HAC  1일 RMSE  1일 변화  1일 CW  중순 물량 계수  중순 물량 t HAC  중순 RMSE  중순 변화  중순 CW(기준 대비)  중순 CW(1일 대비)
계열                                                                                                                                                           
총수출      100.00  365    199   0.84     2.68   0.06   4.08      2.66     2.59  -3.55   3.05      0.12         1.72     2.50  -6.92          3.20          3.44
비반도체      64.72  365    199   0.81     2.69   0.08   5.58      2.92     2.47  -8.09   3.42      0.08         0.88     2.46  -8.79          3.10          1.63
반도체       35.28  365    199   0.85     8.85   0.05   3.22      2.33     8.96   1.23   1.87      0.07         2.21     8.83  -0.23          2.61          3.41
승용차        7.49  365    199   0.61    11.47   0.38  11.89      6.69     9.02 -21.35   4.98     -0.11        -0.71     9.10 -20.65          5.18         -0.65
석유제품       6.11  365    199   0.58     4.90   0.03  

  계열 척도  RMSE    변화    CW
 총수출 명목  2.76  2.93 -0.45
 총수출 물량  2.72  1.52  0.80
비반도체 명목  2.76  2.46 -1.58
비반도체 물량  2.75  2.08  0.71
 반도체 명목  8.90  0.53 -1.20
 반도체 물량  8.83 -0.29  1.24
  계열           기간  기준 RMSE  1일 변화  중순 변화  개월
 총수출    2010~2019      2.4   -1.4   -3.4 120
 총수출    2020~2021      3.2  -15.8  -24.0  24
 총수출 2022~2026.07      3.0   -1.0   -4.3  55
 총수출       코로나 제외      2.6   -1.9   -4.8 189
비반도체    2010~2019      2.4   -3.8   -3.6 120
비반도체    2020~2021      3.9  -14.6  -19.0  24
비반도체 2022~2026.07      2.6  -10.2   -9.4  55
비반도체       코로나 제외      2.5   -6.5   -6.8 189
 반도체    2010~2019      7.9    2.3    0.5 120
 반도체    2020~2021      7.2   -6.0   -5.7  24
 반도체 2022~2026.07     11.1    1.3   -0.1  55
 반도체       코로나 제외      8.9    1.6    0.1 189


   구간   계열  1일 변화  중순 변화
 10,5  총수출   -3.4   -7.0
 10,5 비반도체   -8.1   -8.8
20,10  총수출   -3.6   -6.9
20,10 비반도체   -8.1   -8.8
30,15  총수출   -3.8   -7.1
30,15 비반도체   -8.1   -8.7
20,20  총수출   -3.6   -7.0
20,20 비반도체   -8.0   -8.7
{'제조업 1일': np.float64(-3.505), '제조업 중순': np.float64(-6.834), '제조업 1일 CW': np.float64(3.049), '제조업 중순 CW': np.float64(3.195), '합산 1일 RMSE': np.float64(3.046), '직접 1일 RMSE': np.float64(2.586), '합산 중순 RMSE': np.float64(2.932), '직접 중순 RMSE': np.float64(2.496), '반도체 가중': 0.187, '재구성 상관': np.float64(0.94), '재구성 RMSE 1996~2005': np.float64(5.058), '재구성 RMSE 2006~2015': np.float64(1.616), '재구성 RMSE 2016~2026': np.float64(0.707), '재구성 RMSE 2010~2026': np.float64(0.861)}


## §5. 그림 1·2

그림 1은 총수출의 명목 수출과 물량지수(Törnqvist 연쇄), 광공업생산지수 총지수를 2020=100으로 맞춰 로그 축에 놓은 것이다(12개월 이동평균).
그림 2는 물량지수와 생산지수 $\Delta_{12}$의 60개월 이동 상관이다(총수출, 비반도체, 반도체).

In [10]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False, "figure.dpi": 100})
def style(ax):
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    ax.grid(True, color="0.87", linewidth=0.8); ax.set_axisbelow(True)
def tdate(idx): return pd.to_datetime(pd.Index(idx).astype(int).astype(str), format="%Y%m")
def rebase(s):
    m = s.rolling(12).mean(); return m / m.loc[202001:202012].mean() * 100

def fig_group(g, fname):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(tdate(W.index), rebase(W[g]).values, "--", color="0.55", lw=1.2, label="명목 수출")
    ax.plot(tdate(QT.index), rebase(QT[g]).values, "-", color="black", lw=1.8, label="물량지수")
    ax.plot(tdate(IPX.index), rebase(IPX[g]).values, "-", color="0.55", lw=1.8, label="생산지수")
    ax.set_yscale("log"); ax.set_ylabel("2020=100, 12개월 이동평균 (로그 축)")
    from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator
    ticks = [t for t in (10, 15, 20, 30, 50, 70, 100, 150, 200, 300, 500) if ax.get_ylim()[0] <= t <= ax.get_ylim()[1]]
    ax.yaxis.set_major_locator(FixedLocator(ticks)); ax.yaxis.set_major_formatter(FixedFormatter([str(t) for t in ticks]))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.set_xlim(tdate([199501])[0], tdate([YM1])[0] + pd.Timedelta(days=60))   # 자료 끝(2026.07)에서 자른다
    ax.legend(frameon=False, ncol=4, loc="upper left"); style(ax); fig.tight_layout()
    fig.savefig(os.path.join(IMG, fname), dpi=300); plt.close(fig)
fig_group("총수출", "fig_b1_exports_ip.png")

# 그림 2: 물량 대 생산지수 Δ12의 60개월 이동 상관 — 동조성이 언제 강해지고 약해졌는지는 표로 안 보인다
ROLL = pd.DataFrame({g: eQ[g].rolling(60).corr(eIP[g]) for g in ["총수출", "비반도체", "반도체"]}).dropna(how="all")   # 물량 대 생산, 달력 뺀 잔차
ROLL_R = pd.DataFrame({g: eR[g].rolling(60).corr(eIP[g]) for g in ["총수출", "비반도체", "반도체"]}).dropna(how="all")  # 실질(참고)
ROLL.to_csv(os.path.join(OUT, "tab2_rolling_corr.csv"), encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(10, 4.2))
for g, ls, lw, col in [("총수출", "-", 1.8, "black"), ("비반도체", "--", 1.4, "black"), ("반도체", "-", 1.0, "0.55")]:
    ax.plot(tdate(ROLL.index), ROLL[g].values, ls, lw=lw, color=col, label=g)
ax.axhline(0, color="black", lw=0.8); ax.set_ylim(-0.6, 1.0)
ax.set_ylabel(r"60개월 이동 상관 (수출 물량지수 대 생산지수, $\Delta_{12}$)"); ax.legend(frameon=False, ncol=3, loc="lower left")
style(ax); fig.tight_layout(); fig.savefig(os.path.join(IMG, "fig_b3_rolling_corr.png"), dpi=300); plt.close(fig)
print("그림 저장:", sorted(f for f in os.listdir(IMG) if f.startswith("fig_b")))
print(ROLL.loc[[200012, 200512, 201012, 201512, 201912, 202312, YM1]].round(2).to_string())
print("이동 상관 최소/최대 (총수출):", ROLL["총수출"].idxmin(), round(ROLL["총수출"].min(), 2), ROLL["총수출"].idxmax(), round(ROLL["총수출"].max(), 2))

그림 저장: ['fig_b1_exports_ip.png', 'fig_b3_rolling_corr.png']
         총수출  비반도체   반도체
yyyymm                  
200012  0.03 -0.03  0.17
200512  0.75  0.63  0.44
201012  0.86  0.88  0.30
201512  0.67  0.64  0.29
201912  0.28  0.50 -0.24
202312  0.67  0.81  0.47
202607  0.54  0.24  0.44
이동 상관 최소/최대 (총수출): 200012 0.03 201406 0.9


## §6. 검증

구조 조건은 `assert`로, 본문 인용 수치는 `chk()`로 대조한다. 어긋나면 마지막에 멈춘다. 본문 수치는 집필하면서 채운다.

In [11]:
FAIL = []; N_CHK = [0]
def chk(name, calc, doc, nd=1):
    N_CHK[0] += 1
    c, d = round(float(calc), nd), round(float(doc), nd)
    if abs(c - d) > 10**(-nd) / 2 + 1e-9:
        FAIL.append(f"{name}: 계산 {c} ≠ 본문 {d}")

# 구조
assert R["총수출"].first_valid_index() == 199501 and R["총수출"].notna().all()
assert R["의약품"].first_valid_index() == 201512 and R["제조장비"].first_valid_index() == 200501
assert R["선박"].isna().all()
assert (IPX.loc[:, [g for g in MAP2]].notna().all()).all(), IPX.columns[IPX.isna().any()]
assert (PX["비반도체"] > 0).all() and R["비반도체"].notna().all()
assert all(KW[c] > 0 for a, b, _, _, _ in MAP2.values() for c in a + b)
assert all(XW[c] > 0 for _, _, c, d, _ in MAP2.values() for c in c + d)
# 비반도체 지수가 총지수와 반도체 사이에서 벗어나지 않는 방향으로 움직이는가 (합성의 부호 점검)
assert np.corrcoef(d12(IPX["비반도체"]).dropna(), d12(IPX["총수출"]).loc[d12(IPX["비반도체"]).dropna().index])[0, 1] > 0.8
assert TAB_SYNC2.shape[0] == len(ORDER)
assert TAB_SYNC2.loc["총수출", "물량·생산"] > 0.5, TAB_SYNC2.loc["총수출", "물량·생산"]

# ---- 본문 인용 수치: trend-vs-industrial-production.md (성질별 통계 + 생산지수) ----
TF = TAB_FULL.copy(); TF["명목−생산"] = TF["명목"] - TF["생산지수"]
# 표 3
T3 = {"총수출": [6.5, 4.5, 2.0, 4.4, -2.5, 2.1], "비반도체": [5.6, 3.0, 2.5, 1.9, 0.6, 3.6], "반도체": [9.5, 9.2, 0.3, 19.8, -19.5, -10.3], "화공품": [6.9, 1.4, 5.5, 1.9, 3.6, 5.0], "승용차": [7.7, 1.7, 5.9, 2.3, 3.7, 5.4], "일반기계": [6.3, 1.5, 4.8, 2.8, 2.0, 3.5], "석유제품": [10.2, 5.1, 5.1, 2.1, 3.1, 8.1], "철강제품": [5.1, 1.7, 3.5, 1.5, 2.0, 3.6], "선박": [5.9, 7.3, -1.4, 4.8, -6.2, 1.1], "자동차부품": [10.0, 1.0, 9.0, 5.1, 3.9, 4.9], "무선통신기기": [9.7, 8.5, 1.2, 3.8, -2.7, 5.8], "컴퓨터주변기기": [6.7, 10.9, -4.2, 2.9, -7.0, 3.8], "정밀기기": [8.7, 2.5, 6.2, 2.9, 3.2, 5.8], "이차전지": [9.3, 2.4, 6.9, 7.5, -0.5, 1.9], "제조장비": [27.7, -5.9, 33.6, 2.8, 30.7, 24.9], "의약품": [12.1, 8.1, 4.0, 6.3, -2.3, 5.8], "가전제품": [-0.5, 2.3, -2.9, -0.8, -2.1, 0.2]}
T3 = {k: v for k, v in T3.items() if k in ORDER}   # 논문은 10대 품목과 총수출, 비반도체만 보고한다
for g, vals in T3.items():
    for col, v in zip(["명목", "단가", "물량", "생산지수", "물량−생산", "명목−생산"], vals): chk(f"표3 {g} {col}", TF.loc[g, col], v)
chk("개월 수", len(W), 379, 0); chk("소분류 수", con.execute("SELECT COUNT(*) FROM dim_ksic WHERE level=3").fetchone()[0], 81, 0)
chk("소분류 1995부터", con.execute("SELECT COUNT(*) FROM (SELECT f.ksic, MIN(f.yyyymm) m0 FROM fact_ip f JOIN dim_ksic d USING (ksic) WHERE d.level=3 AND f.measure='prod' GROUP BY 1) WHERE m0=199501").fetchone()[0], 76, 0)
# III.1: 반도체 중량이 3분의 1, 그 대부분이 금액 1%인 기타 개별소자(44307)
chk("III.1 반도체 중량 2024/2023", Q.loc[202401:202412, "반도체"].sum() / Q.loc[202301:202312, "반도체"].sum(), 0.33, 2)
_w = raw[raw.temper_cd == "44307"].groupby(raw.yyyymm // 100)[["dlr", "wgt"]].sum()
_m = raw[raw.temper_cd == "44301"].groupby(raw.yyyymm // 100)[["dlr", "wgt"]].sum()
_Q23, _W23 = Q.loc[202301:202312, "반도체"].sum(), W.loc[202301:202312, "반도체"].sum()
chk("III.1 44307 중량 비중 2023 %", _w.loc[2023, "wgt"] / _Q23 * 100, 87, 0); chk("III.1 44307 금액 비중 2023 %", _w.loc[2023, "dlr"] / _W23 * 100, 2.5)
chk("III.1 메모리 금액 비중 2023 %", _m.loc[2023, "dlr"] / _W23 * 100, 51, 0); chk("III.1 메모리 중량 비중 2023 %", _m.loc[2023, "wgt"] / _Q23 * 100, 2.6)
chk("III.1 44307 중량 2023 만톤", _w.loc[2023, "wgt"] / 1e7, 18.6); chk("III.1 44307 중량 2024 만톤", _w.loc[2024, "wgt"] / 1e7, 4.1)
chk("III.1 메모리 중량 2023 백톤", _m.loc[2023, "wgt"] / 1e5, 56, 0); chk("III.1 메모리 중량 2024 백톤", _m.loc[2024, "wgt"] / 1e5, 55, 0)
# III.1: 총수출 물량 31년 연평균, 끝점 비교 대 월 연쇄
chk("III.1 총수출 물량 끝점", TF.loc["총수출", "물량"], 2.0)
chk("III.1 총수출 물량 월 연쇄", np.log(QT["총수출"].loc[202508:202607].mean() / QT["총수출"].loc[199501:199512].mean()) * 100 / yrs(1995, 2026), 3.0)
# 그림 1 본문 (연평균 수준의 배율)
Qy2 = annual(QT, "mean")
def ratio(T, g, y1, y0): return T.loc[y1, g] / T.loc[y0, g]
chk("생산 2008/1995", ratio(IPy, "총수출", 2008, 1995), 2.5); chk("물량 2008/1995", ratio(Qy2, "총수출", 2008, 1995), 2.1)
chk("생산 2026/2008", ratio(IPy, "총수출", 2026, 2008), 1.6); chk("물량 2026/2008", ratio(Qy2, "총수출", 2026, 2008), 1.2)
chk("생산 2026/2016", ratio(IPy, "총수출", 2026, 2016), 1.21, 2); chk("물량 2026/2016", ratio(Qy2, "총수출", 2026, 2016), 1.12, 2)
chk("명목 2008/1995", ratio(Vy, "총수출", 2008, 1995), 3.4)
# IV.2: 총수출의 Δ12 변동 가운데 달력 설명변수가 설명하는 몫 (%), 단가의 몫
chk("IV.2 달력 몫 총수출 물량", CAL_R2.loc["총수출", "물량"] * 100, 21, 0); chk("IV.2 달력 몫 총수출 명목", CAL_R2.loc["총수출", "명목"] * 100, 4, 0)
chk("IV.2 달력 몫 총수출 단가", CAL_R2_U["총수출"] * 100, 2, 0)
# III.1: 양자간 대 월 연쇄 물량 Δ12
_jq = pd.concat([dQ_["총수출"], dQchain["총수출"]], axis=1).dropna()
chk("III.1 양자간·연쇄 상관", _jq.corr().iloc[0, 1], 0.98, 2); chk("III.1 양자간·연쇄 평균 차이", (_jq.iloc[:, 1] - _jq.iloc[:, 0]).mean() * 100, 0.06, 2)
# 표 4 (달력 효과를 뺀 Δ12 상관과, 차분한 나머지의 교차상관 최대 시차)
T4 = {'총수출': (0.58, 0.63), '비반도체': (0.47, 0.64), '반도체': (0.38, 0.51), '승용차': (0.66, 0.72), '석유제품': (0.56, 0.4), '철강제품': (0.19, 0.41), '컴퓨터주변기기': (0.45, 0.27), '선박': (0.18, 0.18), '무선통신기기': (0.45, 0.48), '자동차부품': (0.54, 0.56), '정밀기기': (0.11, 0.18), '가전제품': (0.42, 0.46)}
for g, vals in T4.items():
    for col, v in zip(["물량·생산", "명목·생산"], vals): chk(f"표4 {g} {col}", TAB_SYNC2.loc[g, col], v, 2)
chk("표4 n", TAB_SYNC2.loc["총수출", "n"], 367, 0)
T4_LAG = {'총수출': (0, 0), '비반도체': (0, 0), '반도체': (0, 0), '승용차': (0, 0), '석유제품': (0, 0), '철강제품': (0, 0), '컴퓨터주변기기': (0, 0), '선박': (7, -10), '무선통신기기': (0, 0), '자동차부품': (0, 0), '정밀기기': (0, 0), '가전제품': (0, 0)}
for g, (lq_, lv_) in T4_LAG.items(): chk(f"표4 {g} 물량 시차", TAB_XC.loc[g, "물량 최대 시차"], lq_, 0); chk(f"표4 {g} 명목 시차", TAB_XC.loc[g, "명목 최대 시차"], lv_, 0)
_dd = TAB_SYNC2["물량·생산"] - TAB_SYNC2["명목·생산"]
if sorted(_dd[_dd > 0.1].index) != sorted(["석유제품", "컴퓨터주변기기"]): FAIL.append("IV.2 물량이 0.1 넘게 높은 계열")
if sorted(_dd[_dd < -0.1].index) != sorted(["비반도체", "반도체", "철강제품"]): FAIL.append("IV.2 명목이 0.1 넘게 높은 계열")
if TAB_SYNC2.loc[TOP10, "물량·생산"].idxmax() != "승용차" or TAB_SYNC2.loc[TOP10, "명목·생산"].idxmax() != "승용차": FAIL.append("IV.2 승용차가 가장 높음")
if list(TAB_SYNC2.loc[TOP10, "물량·생산"].nsmallest(3).index) != ["정밀기기", "선박", "철강제품"]: FAIL.append("IV.2 물량 기준 낮은 셋")
# 그림 2 본문 (이동 상관)
_rt, _rn, _rs = ROLL["총수출"].dropna(), ROLL["비반도체"].dropna(), ROLL["반도체"].dropna(); _rtp = _rt.loc[_rt.idxmax():]
chk("이동상관 총수출 첫 값", _rt.iloc[0], 0.03, 2); chk("이동상관 총수출 첫 시점", _rt.index[0], 200012, 0)
chk("이동상관 총수출 0.5 넘는 첫 달", _rt.index[_rt > 0.5][0], 200304, 0)
chk("이동상관 총수출 최대", _rt.max(), 0.9, 2); chk("이동상관 총수출 최대 시점", _rt.idxmax(), 201406, 0)
chk("이동상관 총수출 최대 뒤 최저", _rtp.min(), 0.24, 2); chk("이동상관 총수출 최대 뒤 최저 시점", _rtp.idxmin(), 202003, 0)
chk("이동상관 총수출 끝", _rt.iloc[-1], 0.54, 2)
chk("이동상관 비반도체 최소", _rn.min(), -0.03, 2); chk("이동상관 비반도체 최소 시점", _rn.idxmin(), 200101, 0)
chk("이동상관 비반도체 0.5 넘는 첫 달", _rn.index[_rn > 0.5][0], 200312, 0)
chk("이동상관 비반도체 최대", _rn.max(), 0.89, 2); chk("이동상관 비반도체 최대 시점", _rn.idxmax(), 202109, 0)
chk("이동상관 비반도체 끝", _rn.iloc[-1], 0.24, 2)
chk("이동상관 반도체 최대", _rs.max(), 0.76, 2); chk("이동상관 반도체 최대 시점", _rs.idxmax(), 201410, 0)
chk("이동상관 반도체 최소", _rs.min(), -0.37, 2); chk("이동상관 반도체 최소 시점", _rs.idxmin(), 202008, 0)
chk("이동상관 반도체 끝", _rs.iloc[-1], 0.44, 2)
# IV.2 선후행: 차분한 나머지의 교차상관
_lq, _lv = TAB_XC["물량 최대 시차"], TAB_XC["명목 최대 시차"]
chk("IV.2 물량 최대 시차 0인 계열 수", (_lq == 0).sum(), 11, 0); chk("IV.2 명목 최대 시차 0인 계열 수", (_lv == 0).sum(), 11, 0)
if list(TAB_XC[(_lq != 0) | (_lv != 0)].index) != ["선박"]: FAIL.append("IV.2 시차 0이 아닌 계열은 선박뿐")
chk("IV.2 선박 물량 최대 상관", TAB_XC.loc["선박", "물량 최대 상관"], 0.17, 2); chk("IV.2 선박 명목 최대 상관", TAB_XC.loc["선박", "명목 최대 상관"], 0.14, 2)
chk("IV.2 총수출 물량 동시", TAB_XC.loc["총수출", "물량 동시 상관"], 0.41, 2); chk("IV.2 총수출 명목 동시", TAB_XC.loc["총수출", "명목 동시 상관"], 0.51, 2)
chk("IV.2 총수출 다른 시차 최대", max(TAB_XC.loc["총수출", "물량 다른 시차 최대"], TAB_XC.loc["총수출", "명목 다른 시차 최대"]), 0.21, 2)
chk("IV.2 대역", TAB_XC.loc["총수출", "대역"], 0.1, 2)
if not (TAB_XC.loc["총수출", "물량 다른 시차 최대"] < TAB_XC.loc["총수출", "물량 동시 상관"] / 2 and TAB_XC.loc["총수출", "명목 다른 시차 최대"] < TAB_XC.loc["총수출", "명목 동시 상관"] / 2): FAIL.append("IV.2 다른 시차는 동시점의 절반 미만")

# ---- 2026-09-14 추가: 본문 산문 수치와 서술의 조건 ----
# IV.1 (2026-09-30): 물량지수가 생산보다 느린/빠른 품목군 수, 가장 느린 셋, 단가 9% 넘는 둘
_gq = TF.loc[TOP10, "물량−생산"]
chk("IV.1 물량이 생산보다 느린 품목군 수", (_gq < 0).sum(), 5, 0); chk("IV.1 빠른 품목군 수", (_gq > 0).sum(), 5, 0)
if list(_gq.nsmallest(3).index) != ["반도체", "컴퓨터주변기기", "선박"]: FAIL.append(f"IV.1 가장 느린 셋 {list(_gq.nsmallest(3).index)}")
if sorted(TF.loc[["반도체", "컴퓨터주변기기", "선박"], "단가"].pipe(lambda x: x[x > 9]).index) != ["반도체", "컴퓨터주변기기"]: FAIL.append("IV.1 단가 9% 넘는 둘")
# II.2: 비반도체 생산지수에서 빠진 코드
_drop_raw = [c for c in EX_ALL if c not in EX_SEMI_CODES]
if _drop_raw != ["C34"]: FAIL.append(f"II.2 원지수 뺀 코드 {_drop_raw}")
# IV.2 표 4: 전체 기간은 Δ12가 있는 1996.01부터
chk("IV.2 표4 Δ12 첫 달", dQ_["총수출"].dropna().index[0], 199601, 0)

# 표 5 (브리지 회귀, 12개 계열): 기준 R2·RMSE, 1일 모형의 명목 계수·HAC t·기준 대비·CW t, 중순 모형의 기준 대비·CW t(1일 대비). 수출 비중은 표 2
T5 = {'총수출': (100.0, 0.84, 2.68, 0.06, 2.7, -4, 3.0, -7, 3.4), '비반도체': (64.7, 0.81, 2.69, 0.08, 2.9, -8, 3.4, -9, 1.6), '반도체': (35.3, 0.85, 8.85, 0.05, 2.3, 1, 1.9, 0, 3.4), '승용차': (7.5, 0.61, 11.47, 0.38, 6.7, -21, 5.0, -21, -0.7), '석유제품': (6.1, 0.58, 4.9, 0.03, 4.1, -2, 2.8, -6, 2.3), '철강제품': (5.3, 0.82, 3.56, 0.01, 0.9, 0, 0.5, 0, -1.0), '컴퓨터주변기기': (3.8, 0.84, 11.54, 0.02, 1.4, 0, 0.5, -1, 2.9), '선박': (3.7, 0.8, 8.45, 0.02, 1.7, 1, 0.9, 1, 0.2), '무선통신기기': (2.2, 0.68, 14.75, 0.13, 3.7, -4, 3.5, -4, -1.0), '자동차부품': (2.1, 0.72, 7.35, 0.11, 3.8, -5, 2.3, -5, -0.2), '정밀기기': (1.2, 0.67, 6.34, 0.03, 2.0, -1, 2.2, 0, -0.7), '가전제품': (0.8, 0.72, 7.1, 0.04, 2.4, -2, 3.4, -2, -0.2)}
_C5 = [("수출 비중", 1), ("기준 R2", 2), ("기준 RMSE", 2), ("명목 계수", 2), ("명목 t HAC", 1), ("1일 변화", 0), ("1일 CW", 1), ("중순 변화", 0), ("중순 CW(1일 대비)", 1)]
for g, v in T5.items():
    for k, (col, nd) in enumerate(_C5): chk(f"{'표2' if col == '수출 비중' else '표5'} {g} {col}", TAB_ITEM.loc[g, col], v[k], nd)
if list(T5) != ORDER or list(TAB_FULL.index) != ORDER or list(TAB_SYNC2.index) != ORDER: FAIL.append("표 행 순서(최근 12개월 수출 비중 순)")
chk("표5 n", TAB_ITEM.loc["총수출", "n"], 365, 0); chk("표5 표본외 n", TAB_ITEM.loc["총수출", "표본외 n"], 199, 0)
chk("IV.3 총수출 명목 t(보통)", TAB_ITEM.loc["총수출", "명목 t"], 4.1, 1)
chk("IV.3 비반도체 중순 추가 감소 %p", TAB_ITEM.loc["비반도체", "1일 변화"] - TAB_ITEM.loc["비반도체", "중순 변화"], 0.7, 1)
chk("IV.3 반도체 중순 감소", abs(TAB_ITEM.loc["반도체", "중순 변화"]), 0.2, 1)
_best = TAB_ITEM.loc[TOP10, ["1일 변화", "중순 변화"]].min(axis=1)
if sorted(_best[_best > -1].index) != sorted(["반도체", "철강제품", "선박", "컴퓨터주변기기", "정밀기기"]): FAIL.append(f"IV.3 1% 넘게 줄지 않는 품목 {list(_best[_best > -1].index)}")
if _best.idxmin() != "승용차": FAIL.append("IV.3 승용차가 가장 큼")
if not all(TAB_ITEM.loc[g, "1일 CW"] > 1.645 for g in ["총수출", "비반도체", "자동차부품", "무선통신기기", "석유제품", "가전제품", "승용차"]): FAIL.append("IV.3 유의하다고 쓴 품목의 1일 CW")
if not (TAB_ITEM.loc["총수출", "중순 CW(1일 대비)"] > 1.645): FAIL.append("IV.3 총지수 중순 추가 정보 유의")
# 전월 수출
_lg = TAB_LAG.set_index(["계열", "척도"])
chk("IV.3 총수출 전월 명목 변화", _lg.loc[("총수출", "명목"), "변화"], 2.9, 1); chk("IV.3 총수출 전월 물량 변화", _lg.loc[("총수출", "물량"), "변화"], 1.5, 1)
if not (TAB_LAG["CW"].max() < 1.282): FAIL.append("IV.3 전월 수출 CW가 1.28 미만")
if not (TAB_LAG["변화"] > -1).all(): FAIL.append("IV.3 전월 수출이 오차를 줄이지 않음")
# 표 6 (하위 기간)
T6 = {'총수출|2010~2019': (120, 2.42, -1, -3), '총수출|2020~2021': (24, 3.18, -16, -24), '총수출|2022~2026.07': (55, 2.98, -1, -4), '총수출|코로나 제외': (189, 2.58, -2, -5), '비반도체|2010~2019': (120, 2.4, -4, -4), '비반도체|2020~2021': (24, 3.89, -15, -19), '비반도체|2022~2026.07': (55, 2.65, -10, -9), '비반도체|코로나 제외': (189, 2.53, -7, -7), '반도체|2010~2019': (120, 7.94, 2, 1), '반도체|2020~2021': (24, 7.16, -6, -6), '반도체|2022~2026.07': (55, 11.12, 1, 0), '반도체|코로나 제외': (189, 8.89, 2, 0)}
_sb = TAB_SUB.set_index(["계열", "기간"])
for key, (n_, rm, d1, d2) in T6.items():
    g, k = key.split("|"); r_ = _sb.loc[(g, k)]
    chk(f"표6 {key} 개월", r_["개월"], n_, 0); chk(f"표6 {key} 기준 RMSE", r_["기준 RMSE"], rm, 2); chk(f"표6 {key} 1일", r_["1일 변화"], d1, 0); chk(f"표6 {key} 중순", r_["중순 변화"], d2, 0)
# 강건성
ROBC = {'제조업 1일': -3.51, '제조업 중순': -6.83, '제조업 1일 CW': 3.05, '제조업 중순 CW': 3.2, '합산 1일 RMSE': 3.05, '직접 1일 RMSE': 2.59, '합산 중순 RMSE': 2.93, '직접 중순 RMSE': 2.5, '반도체 가중': 0.19, '재구성 상관': 0.94, '재구성 RMSE 1996~2005': 5.06, '재구성 RMSE 2006~2015': 1.62, '재구성 RMSE 2016~2026': 0.71, '재구성 RMSE 2010~2026': 0.86}
for k in ["제조업 1일", "제조업 중순", "합산 1일 RMSE", "직접 1일 RMSE", "재구성 상관"]: chk(f"강건성 {k}", ROB[k], ROBC[k], 2)
for k in ["재구성 RMSE 1996~2005", "재구성 RMSE 2006~2015", "재구성 RMSE 2016~2026", "재구성 RMSE 2010~2026"]: chk(f"II.2 {k}", ROB[k], ROBC[k], 1)
_cw = TAB_CALW
for g, c, lo, hi in [("총수출", "1일 변화", -4.0, -3.0), ("총수출", "중순 변화", -7.0, -7.0), ("비반도체", "1일 변화", -8.0, -8.0), ("비반도체", "중순 변화", -9.0, -9.0)]:
    chk(f"강건성 달력 {g} {c} 최소", _cw[_cw.계열 == g][c].min(), lo, 0); chk(f"강건성 달력 {g} {c} 최대", _cw[_cw.계열 == g][c].max(), hi, 0)

print(f"검증 {N_CHK[0]}건, 어긋남 {len(FAIL)}건")
for f in FAIL: print("  ", f)
assert not FAIL

검증 357건, 어긋남 0건
